<div align="center">

# **_Olist Revenue Intelligence — Build Analytical Dataset_**

### A structured transformation of the Olist relational schema into item-level and order-level analytical datasets for revenue analysis, operational execution tracking, and downstream business intelligence.

</div>

## 1. Notebook Objectives

The goal of this notebook is to transform the raw Olist relational tables into clean analytical datasets that can support downstream business analysis, SQL KPI construction, dashboard development, and future predictive extensions.

More specifically, this notebook aims to:

- restrict the analytical scope to **delivered orders only**
- build a clean **item-level analytical table** for revenue and portfolio analysis
- build a clean **order-level analytical table** for delivery KPIs, review-based feedback analysis, and future predictive modeling
- ensure that the analytical grain is aligned with the nature of the variables used downstream

Two complementary datasets will therefore be created:

- **fact_order_items**: one row per order item
- **fact_orders**: one row per order

In [55]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 2. Load Raw Data

Only the tables required for the analytical datasets are loaded in this notebook.

In [56]:
raw_path = "../data/raw/"
processed_path = "../data/processed/"

orders = pd.read_csv(raw_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(raw_path + "olist_order_items_dataset.csv")
customers = pd.read_csv(raw_path + "olist_customers_dataset.csv")
products = pd.read_csv(raw_path + "olist_products_dataset.csv")
translation = pd.read_csv(raw_path + "product_category_name_translation.csv")
reviews = pd.read_csv(raw_path + "olist_order_reviews_dataset.csv")

## 3. Parse Date Variables

Relevant date columns are converted to datetime format before any analytical transformation.

In [57]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

review_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

## 4. Restrict the Analytical Scope

The retained analytical population is limited to **delivered orders only**, in line with the methodological decisions established in Notebook 1.

In [58]:
orders_delivered = orders[orders["order_status"] == "delivered"].copy()

print("Delivered orders:", len(orders_delivered))
print("Unique delivered order_id:", orders_delivered["order_id"].nunique())

Delivered orders: 96478
Unique delivered order_id: 96478


## 5. Prepare Enrichment Layers

Customer and product information are prepared separately before being merged into the analytical tables.

In [59]:
customer_features = customers[[
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
]].copy()

In [60]:
products_enriched = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products_enriched = products_enriched[[
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]].copy()

## 6. Build the Item-Level Analytical Table

The first analytical table is built at the **order item level**.

This table is designed to support:
- revenue structure analysis
- category analysis
- seller analysis
- portfolio analysis

In [61]:
order_items_work = order_items.copy()

order_items_work["item_revenue"] = order_items_work["price"]
order_items_work["total_revenue"] = (
    order_items_work["price"] + order_items_work["freight_value"]
)

In [62]:
fact_order_items = (
    order_items_work
    .merge(
        orders_delivered[[
            "order_id",
            "customer_id",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]],
        on="order_id",
        how="inner"
    )
    .merge(customer_features, on="customer_id", how="left")
    .merge(products_enriched, on="product_id", how="left")
)

In [63]:
fact_order_items["order_month"] = (
    fact_order_items["order_purchase_timestamp"].dt.to_period("M").astype(str)
)
fact_order_items["order_year"] = fact_order_items["order_purchase_timestamp"].dt.year

### Methodological note

Delivery-related variables such as:
- `delivery_lead_time_days`
- `estimated_vs_actual_days`
- `is_late`

are naturally defined at the **order level**.

To preserve a clean analytical separation, they are kept in the **order-level analytical table only**, rather than repeated in the item-level dataset.

In [64]:
print("fact_order_items rows:", len(fact_order_items))
print("Unique orders in fact_order_items:", fact_order_items["order_id"].nunique())
print("Unique customers in fact_order_items:", fact_order_items["customer_unique_id"].nunique())

fact_order_items rows: 110197
Unique orders in fact_order_items: 96478
Unique customers in fact_order_items: 93358


## 7. Build the Order-Level Analytical Table

The second analytical table is built at the **order level**.

This table is designed to support:
- delivery KPI analysis
- lateness analysis
- review-based feedback analysis
- future predictive modeling

In [65]:
order_level_metrics = (
    order_items_work
    .groupby("order_id", as_index=False)
    .agg(
        order_revenue=("total_revenue", "sum"),
        n_items=("order_item_id", "count"),
        n_sellers=("seller_id", "nunique")
    )
)

In [66]:
order_category_metrics = (
    order_items
    .merge(
        products_enriched[["product_id", "product_category_name_english"]],
        on="product_id",
        how="left"
    )
    .groupby("order_id", as_index=False)
    .agg(
        n_categories=("product_category_name_english", "nunique")
    )
)

In [67]:
fact_orders = (
    orders_delivered
    .merge(customer_features, on="customer_id", how="left")
    .merge(order_level_metrics, on="order_id", how="left")
    .merge(order_category_metrics, on="order_id", how="left")
)

In [68]:
fact_orders["order_month"] = (
    fact_orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
)
fact_orders["order_year"] = fact_orders["order_purchase_timestamp"].dt.year

fact_orders["delivery_lead_time_days"] = (
    fact_orders["order_delivered_customer_date"]
    - fact_orders["order_purchase_timestamp"]
).dt.days

fact_orders["estimated_vs_actual_days"] = (
    fact_orders["order_delivered_customer_date"]
    - fact_orders["order_estimated_delivery_date"]
).dt.days

fact_orders["is_late"] = (
    fact_orders["estimated_vs_actual_days"] > 0
).astype(int)

## 8. Build the Order-Level Review Layer

Reviews are naturally linked to `order_id`, so customer feedback must be aggregated at the **order level** before being merged into the analytical table.

In [69]:
reviews_order_level = (
    reviews
    .sort_values(["order_id", "review_answer_timestamp"])
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        n_reviews=("review_id", "count"),
        review_creation_date=("review_creation_date", "max"),
        review_answer_timestamp=("review_answer_timestamp", "max")
    )
)

In [70]:
reviews_order_level["has_review"] = 1

reviews_order_level["is_low_review"] = (
    reviews_order_level["review_score"] < 3
).astype(int)

reviews_order_level["is_neutral_review"] = (
    (reviews_order_level["review_score"] >= 3)
    & (reviews_order_level["review_score"] < 4)
).astype(int)

reviews_order_level["is_high_review"] = (
    reviews_order_level["review_score"] >= 4
).astype(int)

In [71]:
reviews_flag_check = (
    reviews_order_level[[
        "is_low_review",
        "is_neutral_review",
        "is_high_review"
    ]]
    .sum(axis=1)
)

reviews_flag_check.value_counts()

1    98673
Name: count, dtype: int64

In [72]:
assert (reviews_flag_check == 1).all(), "Review band assignment is inconsistent"

## 9. Merge Reviews into the Order-Level Analytical Table

In [73]:
fact_orders = fact_orders.merge(
    reviews_order_level,
    on="order_id",
    how="left"
)

fact_orders["has_review"] = fact_orders["has_review"].fillna(0).astype(int)
fact_orders["n_reviews"] = fact_orders["n_reviews"].fillna(0).astype(int)

fact_orders["is_low_review"] = fact_orders["is_low_review"].fillna(0).astype(int)
fact_orders["is_neutral_review"] = fact_orders["is_neutral_review"].fillna(0).astype(int)
fact_orders["is_high_review"] = fact_orders["is_high_review"].fillna(0).astype(int)

In [74]:
review_flag_check = (
    fact_orders.loc[fact_orders["has_review"] == 1, [
        "is_low_review",
        "is_neutral_review",
        "is_high_review"
    ]]
    .sum(axis=1)
)

review_flag_check.value_counts()

1    95832
Name: count, dtype: int64

In [75]:
assert len(fact_orders) == fact_orders["order_id"].nunique(), "fact_orders is no longer one row per order"
assert (review_flag_check == 1).all(), "Review band assignment is inconsistent"

## 10. Final Quality Checks

In [76]:
print("fact_order_items rows:", len(fact_order_items))
print("Unique orders in fact_order_items:", fact_order_items["order_id"].nunique())
print("Unique customers in fact_order_items:", fact_order_items["customer_unique_id"].nunique())

print()

print("fact_orders rows:", len(fact_orders))
print("Unique order_id in fact_orders:", fact_orders["order_id"].nunique())
print("Review coverage (%):", round(fact_orders["has_review"].mean() * 100, 2))
print("Average review score:", round(fact_orders["review_score"].mean(), 2))

fact_order_items rows: 110197
Unique orders in fact_order_items: 96478
Unique customers in fact_order_items: 93358

fact_orders rows: 96478
Unique order_id in fact_orders: 96478
Review coverage (%): 99.33
Average review score: 4.16


In [77]:
fact_orders[[
    "has_review",
    "is_low_review",
    "is_neutral_review",
    "is_high_review"
]].mean() * 100

has_review           99.330417
is_low_review        12.714816
is_neutral_review     8.228819
is_high_review       78.386782
dtype: float64

### Interpretation of the final quality checks

The final checks confirm that both analytical tables were built as intended:

- `fact_order_items` preserves the item-level grain required for revenue and portfolio analysis
- `fact_orders` preserves a strict order-level grain, with one row per delivered order
- review coverage is very high within the retained analytical scope
- the average review score indicates that post-purchase feedback is globally positive in the delivered-order population

These checks validate the structural consistency of the analytical datasets before downstream business analysis.

## 11. Export Analytical Tables

The two analytical datasets are exported to the processed data folder for downstream use in Python analysis, SQL KPI construction, Power BI development, and future predictive work.

In [78]:
fact_order_items.to_csv(processed_path + "fact_order_items.csv", index=False)
fact_orders.to_csv(processed_path + "fact_orders.csv", index=False)

print("Saved: fact_order_items.csv")
print("Saved: fact_orders.csv")

Saved: fact_order_items.csv
Saved: fact_orders.csv


## 12. Notebook Conclusion

This notebook transformed the raw Olist relational tables into two complementary analytical datasets.

### Outputs created
- **fact_order_items**: item-level analytical table for revenue structure and portfolio analysis
- **fact_orders**: order-level analytical table for delivery KPIs, review-based feedback analysis, and future predictive modeling

### Main methodological principles preserved
- delivered orders only
- revenue defined as `price + freight_value`
- customer identity tracked through `customer_unique_id`
- strict separation between item-level and order-level analytical concepts

These datasets now provide the analytical foundation for the next notebook, which will focus on revenue intelligence analysis, operational execution patterns, customer feedback signals, and downstream business interpretation.

In [79]:
fact_order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_revenue,total_revenue,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,product_category_name,product_category_name_english,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,order_month,order_year
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,58.90,72.19,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,cool_stuff,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,2017-09,2017
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,239.90,259.83,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,pet_shop,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,2017-04,2017
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,199.00,216.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,moveis_decoracao,furniture_decor,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,2018-01,2018
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,12.99,25.78,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,af861d436cfc08b2c2ddefd0ba074622,atibaia,SP,perfumaria,perfumery,42.0,480.0,1.0,200.0,16.0,10.0,15.0,2018-08,2018
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,199.90,218.04,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,ferramentas_jardim,garden_tools,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,2017-02,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110192,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,299.99,343.40,b51593916b4b8e0d6f66f2ae24f2673d,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18,0c9aeda10a71f369396d0c04dce13a64,sao luis,MA,utilidades_domesticas,housewares,43.0,1002.0,3.0,10150.0,89.0,15.0,40.0,2018-04,2018
110193,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,350.00,386.53,84c5d4fbaf120aae381fad077416eaa0,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01,0da9fe112eae0c74d3ba1fe16de0988b,curitiba,PR,informatica_acessorios,computers_accessories,31.0,232.0,1.0,8950.0,45.0,26.0,38.0,2018-07,2018
110194,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,99.90,116.85,29309aa813182aaddc9b259e31b870e6,2017-10-23 17:07:56,2017-10-24 17:14:25,2017-10-26 15:13:14,2017-10-28 12:22:22,2017-11-10,cd79b407828f02fdbba457111c38e4c4,sao paulo,SP,esporte_lazer,sports_leisure,43.0,869.0,1.0,967.0,21.0,24.0,19.0,2017-10,2017
110195,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,55.

In [80]:
fact_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,order_revenue,n_items,n_sellers,n_categories,order_month,order_year,delivery_lead_time_days,estimated_vs_actual_days,is_late,review_score,n_reviews,review_creation_date,review_answer_timestamp,has_review,is_low_review,is_neutral_review,is_high_review
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,38.71,1,1,1,2017-10,2017,8.0,-8.0,0,4.0,1,2017-10-11,2017-10-12 03:43:48,1,0,0,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,barreiras,BA,141.46,1,1,1,2018-07,2018,13.0,-6.0,0,4.0,1,2018-08-08,2018-08-08 18:37:50,1,0,0,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,179.12,1,1,1,2018-08,2018,9.0,-18.0,0,5.0,1,2018-08-18,2018-08-22 19:07:58,1,0,0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,72.20,1,1,1,2017-11,2017,13.0,-13.0,0,5.0,1,2017-12-03,2017-12-05 19:21:58,1,0,0,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,28.62,1,1,1,2018-02,2018,2.0,-10.0,0,5.0,1,2018-02-17,2018-02-18 13:02:51,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96473,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,6359f309b166b0196dbf7ad2ac62bb5a,sao jose dos campos,SP,85.08,1,1,1,2017-03,2017,8.0,-11.0,0,5.0,1,2017-03-22,2017-03-23 11:02:08,1,0,0,1
96474,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,da62f9e57a76d978d02ab5362c509660,praia grande,SP,195.00,1,1,1,2018-02,2018,22.0,-2.0,0,4.0,1,2018-03-01,2018-03-02 17:50:01,1,0,0,1
96475,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,737520a9aad80b3fbbdad19b66b37b30,nova vicosa,BA,271.01,1,1,1,2017-08,2017,24.0,-6.0,0,5.0,1,2017-09-22,2017-09-22 23:10:57,1,0,0,1
96476,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,5097a5312c8b157bb7be58ae360ef43c,japuiba,RJ,441.16,2,1,1,2018-01,2018,17.0,-21.0,0,2.0,1,2018-01-26,2018-01-27 09:16:56,1,1,0,0
